In [8]:
import pandas as pd
import sqlite3

## 1. Connection

In [9]:
db = sqlite3.connect("../data/checking-logs.sqlite")

## 2, 3. Creating dataframes

In [10]:
sql_test = """
WITH diffs AS (
  SELECT
    t.uid,
    (strftime('%s', t.first_commit_ts) - d.deadlines)/3600.0 AS diff_h,
    CASE
      WHEN strftime('%s', t.first_commit_ts) < strftime('%s', t.first_view_ts) THEN 'before'
      ELSE 'after'
    END AS time
  FROM test AS t
  JOIN deadlines AS d ON d.labs = t.labname
  WHERE t.labname != 'project1'
),
eligible AS (
  SELECT uid
  FROM diffs
  GROUP BY uid
  HAVING SUM(time='before') > 0 AND SUM(time='after') > 0
),
user_avgs AS (
  SELECT
    uid,
    AVG(CASE WHEN time='before' THEN diff_h END) AS before_avg,
    AVG(CASE WHEN time='after'  THEN diff_h END) AS after_avg
  FROM diffs
  WHERE uid IN (SELECT uid FROM eligible)
  GROUP BY uid
)
SELECT 'before' AS time, AVG(before_avg) AS avg_diff FROM user_avgs
UNION ALL
SELECT 'after'  AS time, AVG(after_avg ) AS avg_diff FROM user_avgs;
"""
test_results = pd.io.sql.read_sql(sql_test, db)[["time","avg_diff"]]
test_results

,time,avg_diff
0,before,-66.679583
1,after,-100.178181


In [ ]:
sql_control = """
WITH diffs AS (
  SELECT
    c.uid,
    (strftime('%s', c.first_commit_ts) - d.deadlines)/3600.0 AS diff_h,
    CASE
      WHEN strftime('%s', c.first_commit_ts) < strftime('%s', c.first_view_ts) THEN 'before'
      ELSE 'after'
    END AS time
  FROM control AS c
  JOIN deadlines AS d ON d.labs = c.labname
  WHERE c.labname != 'project1'
),
all_exists AS (
  SELECT uid
  FROM diffs
  GROUP BY uid
  HAVING SUM(time='before') > 0 AND SUM(time='after') > 0
),
user_avgs AS (
  SELECT
    uid,
    AVG(CASE WHEN time='before' THEN diff_h END) AS before_avg,
    AVG(CASE WHEN time='after'  THEN diff_h END) AS after_avg
  FROM diffs
  WHERE uid IN (SELECT uid FROM all_exists)
  GROUP BY uid
)
SELECT 'before' AS time, AVG(before_avg) AS avg_diff FROM user_avgs
UNION ALL
SELECT 'after'  AS time, AVG(after_avg ) AS avg_diff FROM user_avgs;
"""
control_results = pd.io.sql.read_sql(sql_control, db)[["time","avg_diff"]]
control_results

,time,avg_diff
0,before,-98.467852
1,after,-99.803549


## 4. Closing connection

In [12]:
db.close()

## 5. ANSWER

- Да, гипотеза подтвердилась
в тестовой группе (мы в ней не приводили время просмотра к среднему значению) есть существенная разница во времени, оно увеличилось на почти на 33 часа
а в контрольной выборке у нас нет никаких изменений, потому что они и не заходили на сайт вовсе!